# 📌 MODUL 03 — Pertemuan 04: Apache Spark & PySpark DataFrame Dasar
> **Tujuan Pembelajaran:** Memahami mengapa Apache Spark menggantikan Hadoop MapReduce, menguasai konsep **Lazy Evaluation**, serta mempraktikkan operasi dasar PySpark DataFrame (`select`, `filter`, `withColumn`, `groupBy`, dan `agg`).

## ⚡ 1. Mengapa Apache Spark Jauh Lebih Cepat dari Hadoop MapReduce?

| Karakteristik | Hadoop MapReduce | Apache Spark |
|---|---|---|
| **Penyimpanan Antara** | Harddisk (I/O disk lambat) | Memori RAM (**In-Memory Computing**, 100x lebih cepat) |
| **Model Eksekusi** | Linear kaku (Map -> Tulis Disk -> Reduce) | Berbasis **DAG** (*Directed Acyclic Graph*) |
| **Abstraksi Data** | Teks mentah berbasis Key-Value | **DataFrame** dengan skema tabel terstruktur |
| **Optimasi Kueri** | Manual oleh programmer | Otomatis oleh **Catalyst Optimizer** & Tungsten |

## 💡 2. Konsep Super Penting: *Lazy Evaluation* vs *Eager Execution*

> **Lazy Evaluation:**  
> Spark **TIDAK** langsung mengeksekusi perhitungan saat kode ditulis. Spark hanya mencatat rencana alur kerja ke dalam graf DAG. Eksekusi fisik nyata **BARU BERJALAN** saat ada perintah bertipe **Action** dipanggil!

- **Transformation (Lazy):** `select()`, `filter()`, `withColumn()`, `drop()`, `groupBy()`, `join()`, `orderBy()`.
- **Action (Eager):** `show()`, `count()`, `collect()`, `first()`, `take()`, `write.parquet()`, `write.csv()`.

## ⚙️ 3. Inisialisasi SparkSession (Arch Linux Zero-Conflict)
Selalu jalankan sel ini pertama kali untuk mengunci runtime Java 11 dan IP lokal:

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Buat SparkSession
spark = SparkSession.builder \
    .appName("Responsi-Pertemuan04") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession Aktif dan Siap Digunakan!")

## 📊 4. Memuat Data ke PySpark DataFrame
Mari kita buat dataset transaksi e-commerce langsung sebagai Spark DataFrame:

In [ ]:
data_transaksi = [
    ("TRX-01", "Magelang", "Elektronik", 2, 750000),
    ("TRX-02", "Magelang", "Fashion", 4, 120000),
    ("TRX-03", "Yogyakarta", "Elektronik", 1, 1500000),
    ("TRX-04", "Yogyakarta", "Makanan", 15, 25000),
    ("TRX-05", "Semarang", "Elektronik", 3, 850000),
    ("TRX-06", "Semarang", "Fashion", 5, 95000),
    ("TRX-07", "Solo", "Makanan", 20, 15000),
    ("TRX-08", "Purworejo", "Elektronik", 2, 1200000),
    ("TRX-09", "Magelang", "Makanan", 8, 30000),
    ("TRX-10", "Purworejo", "Fashion", 3, 110000),
]
columns = ["order_id", "kota", "kategori", "unit_terjual", "harga_satuan"]

df = spark.createDataFrame(data_transaksi, schema=columns)
print("Skema Tabel:")
df.printSchema()

print()
print("Isi 5 Baris Pertama:")
df.show(5)

## 🔍 5. Operasi Kolom: `select()` dan `filter()`
Menyaring baris data dan memilih kolom tertentu:

In [ ]:
# 1. Memilih kolom tertentu
df_pilih = df.select("order_id", "kota", "unit_terjual")
df_pilih.show(3)

# 2. Filtering kondisi ganda: unit_terjual >= 3 DAN kota bukan Semarang
# PENTING: Gunakan operator & (AND), | (OR), dan wajib tanda kurung ()
df_disaring = df.filter((F.col("unit_terjual") >= 3) & (F.col("kota") != "Semarang"))
print("Hasil Filter Data:")
df_disaring.show()

## ➕ 6. Menambah Kolom Baru: `withColumn()` & `when().otherwise()`
Menghitung total pendapatan dan mengelompokkan nilai transaksi (kondisional IF-ELSE):

In [ ]:
# 1. Menambahkan kolom total pendapatan (unit x harga)
df_olah = df.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga_satuan"))

# 2. Menambahkan kategori transaksi menggunakan when-otherwise
df_olah = df_olah.withColumn(
    "status_omset",
    F.when(F.col("total_pendapatan") >= 1000000, "Omset Besar")
     .when(F.col("total_pendapatan") >= 300000, "Omset Menengah")
     .otherwise("Omset Kecil")
)

df_olah.show()

## 📈 7. Agregasi Data: `groupBy()` dan `agg()`
Meringkas kinerja penjualan per kota (total omset, rata-rata belanja, banyak transaksi):

In [ ]:
df_performa = df_olah.groupBy("kota").agg(
    F.sum("total_pendapatan").alias("total_omset"),
    F.round(F.avg("total_pendapatan"), 0).alias("rata_rata_omset"),
    F.count("order_id").alias("banyak_transaksi"),
    F.max("total_pendapatan").alias("transaksi_terbesar")
).orderBy(F.col("total_omset").desc())

print("Performa Penjualan Cabang per Kota:")
df_performa.show()

## 💾 8. Action: Menyimpan Data ke Format CSV
Menyimpan ringkasan data hasil analisis ke disk/HDFS:

In [ ]:
output_path = "output_latihan_pertemuan4"
df_performa.write.mode("overwrite").option("header", "true").csv(output_path)
print(f"✅ Data berhasil disimpan ke: {output_path}")

# Verifikasi pembacaan kembali
df_baca = spark.read.option("header", "true").csv(output_path)
print(f"Data terbaca kembali: {df_baca.count()} baris.")

# Bersihkan folder latihan
!rm -rf output_latihan_pertemuan4

## 🛑 9. Menutup SparkSession
Selalu tutup sesi di akhir notebook untuk melepaskan resource CPU dan RAM:

In [ ]:
spark.stop()
print("🛑 SparkSession berhasil dihentikan!")